In [1]:
import csv
import requests
import pandas as pd

In [5]:
# Ranking URL to get latest (Live) Men's FIFA rankings
RANKINGS_URL = 'https://api.fifa.com/api/v3/fifarankings/rankings/live?gender=1&sportType=0&language=en'
RANKINGS_URL = 'https://api.fifa.com/api/v3/fifarankings/rankings/live?gender=1&sportType=0&language=en'

def fetch_rankings():
    """Fetch FIFA men's rankings from FIFA's internal rankings endpoint."""
    headers = {
        'Accept': 'application/json',
        'User-Agent': (
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/125.0 Safari/537.36'
        ),
        'Origin': 'https://inside.fifa.com/',
        'Referer': 'https://inside.fifa.com/',
    }

    response = requests.get(RANKINGS_URL, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()


def normalise_row(row, date=None):
    """Convert one FIFA ranking row into a flat CSV-style dictionary."""
    team = row.get('team', row)
    country_code = (
        team.get('abbreviation')
        or team.get('countryCode')
        or team.get('code')
        or row.get('countryCode')
    )

    country_name = (
        team.get('name')
        or team.get('countryName')
        or row.get('countryName')
        or row.get('name')
    )

    return {
        'date': date,
        'country': country_name,
        'rank': row.get('rank'),
        'previous_rank': row.get('previousRank') or row.get('previous_rank'),
        'total_points': row.get('totalPoints') or row.get('points'),
        'previous_points': row.get('previousPoints') or row.get('previous_points'),
        'flagUrl': f'https://api.fifa.com/api/v3/picture/flags-sq-2/{country_code}',
        'countryUrl': f'https://inside.fifa.com/fifa-world-ranking/{country_code}?gender=men',
        'conf': row.get('confederation') or team.get('confederation'),
    }

In [6]:
import pandas as pd


def get_localized_description(values, locale='en-GB'):
    """Extract a localized Description from FIFA-style locale dictionaries."""
    if not isinstance(values, list):
        return None

    for item in values:
        if item.get('Locale') == locale:
            return item.get('Description')

    return values[0].get('Description') if values else None


def rankings_to_dataframe(data, locale='en-GB', gender='men'):
    """Parse FIFA ranking JSON into the target table schema."""
    rows = []
    ranking_date = pd.Timestamp.now().strftime('%Y-%m-%d')

    for item in data.get('Results', []):
        country_code = item.get('IdCountry')

        rows.append({
            'date': ranking_date,
            'country': get_localized_description(item.get('TeamName'), locale=locale),
            'rank': item.get('Rank'),
            'previous_rank': item.get('PrevRank'),
            'total_points': item.get('TotalPoints'),
            'previous_points': item.get('PrevPoints'),
            'flagUrl': (
                f'https://api.fifa.com/api/v3/picture/flags-sq-2/{country_code}'
                if country_code else None
            ),
            'countryUrl': (
                f'https://inside.fifa.com/fifa-world-ranking/{country_code}'
                if country_code else None
            ),
            'conf': item.get('ConfederationName'),
        })

    columns = [
        'date',
        'country',
        'rank',
        'previous_rank',
        'total_points',
        'previous_points',
        'flagUrl',
        'countryUrl',
        'conf',
    ]

    return pd.DataFrame(rows, columns=columns)

In [7]:
data = fetch_rankings()
df = rankings_to_dataframe(data)
date = pd.Timestamp.now().strftime('%Y-%m-%d')
df.to_csv(f'data/raw/fifa_world_rankings_{date}.csv', index=False)